# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library, based on its Croissant schema.

### Dataset Source
The dataset schema is provided by a Croissant JSON-LD file at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install the required library
!pip install mlcroissant

## 1. Data Loading
We use `mlcroissant` to load the Croissant metadata and access the records in the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access high-level metadata
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', '')}")
print(f"Description: {getattr(metadata, 'description', '')}")
print(f"License: {getattr(metadata, 'license', '')}")
print(f"Citation: {getattr(metadata, 'citeAs', '')}")

## 2. Data Overview
Review available record sets and their fields, all referenced by `@id`.

In [ ]:
# List all record sets within the dataset by @id and their fields
if hasattr(dataset, 'record_sets'):
    available_record_sets = [r for r in dataset.record_sets]
    print("Available record sets:")
    for rs in available_record_sets:
        rid = getattr(rs, '@id', None) or getattr(rs, 'id', None)
        print(f"  Record Set @id: {rid}")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for fld in rs.fields:
                f_id = getattr(fld, '@id', None) or getattr(fld, 'id', None)
                f_name = getattr(fld, 'name', f_id)
                print(f"      - {f_name} (@id={f_id})")
        else:
            print("    No fields found.")
else:
    print("No record_sets attribute found on loaded dataset.")

## 3. Data Extraction
Let's extract data from the main record set(s) into pandas DataFrames using their `@id`.

_Note: Use IDs as shown in the overview above (matching `@id` fields exactly)._

In [ ]:
# Identify the main record set(s). Most clinical datasets bundle all tabular data into one record set.
record_sets = [rs for rs in getattr(dataset, 'record_sets', [])]

# Use the @id of the main record set for extraction
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
print("Extracting record set(s):", record_set_ids)

dataframes = {}
for rs_id in record_set_ids:
    if not rs_id:
        continue
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records for record set {rs_id}")

# Display column names of the first record set
if record_set_ids:
    main_rs_id = record_set_ids[0]
    if main_rs_id in dataframes:
        print(f"\nColumns for record set {main_rs_id}:")
        pprint(dataframes[main_rs_id].columns.tolist())
        display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Perform preliminary analysis using `@id` references for columns (fields). We'll select a numeric field, filter data, normalize columns, and group by key attributes.

In [ ]:
# Pick the main dataframe (first record set)
main_rs_id = record_set_ids[0] if record_set_ids else None
df = dataframes[main_rs_id] if main_rs_id in dataframes else None

if df is not None:
    # Find numeric columns by trying to convert to numeric
    from pandas.api.types import is_numeric_dtype
    numeric_fields = [
        col for col in df.columns if is_numeric_dtype(df[col])
    ]
    if not numeric_fields:
        # Try by coercing each column
        numeric_fields = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notnull().any()]

    print("Numeric fields:", numeric_fields)

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")

        # Apply filter (example threshold: mean of numeric field)
        threshold = df[numeric_field_id].astype(float).mean()
        filtered_df = df[df[numeric_field_id].astype(float) > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean): {len(filtered_df)} rows")

        # Normalize the field
        mean = filtered_df[numeric_field_id].astype(float).mean()
        std = filtered_df[numeric_field_id].astype(float).std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - mean) / std
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a potential grouping field (categorical)
        categorical_fields = [
            col for col in df.columns
            if (df[col].nunique() < 10 and df[col].nunique() > 1 and not is_numeric_dtype(df[col])) or 'sex' in col.lower() or 'site' in col.lower() or 'group' in col.lower()
        ]
        if categorical_fields:
            group_field_id = categorical_fields[0]
            print(f"Grouping by {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print("Grouped means:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field for grouping found.")
    else:
        print("No numeric field identified in data frame.")
else:
    print("Main data frame not found or empty.")

## 5. Visualization

Generate visualizations of data distributions or relationships using appropriate fields (referenced by their `@id` columns).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_fields:
    # Histogram for numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].astype(float), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If grouping field exists, boxplot
    if 'group_field_id' in locals():
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id].astype(float))
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to explore the FAIR² dataset on second primary colorectal cancer. We examined its metadata, inspected available record sets and fields via their `@id`, loaded tabular data into DataFrames, performed example filtering/normalization/grouping, and visualized relevant distributions. This workflow facilitates reproducible FAIR data exploration and is fully referenceable using Croissant schema identifiers.